# Transformer Solver for the 1-D Poisson Equation
## Learning the Inverse Differential Operator with PyTorch

This notebook trains an encoder-only **Transformer** to approximate the inverse
of the 1-D Poisson operator $\mathcal{L} = -d^2/dx^2$ with Dirichlet boundary
conditions, and compares the result against Thomas' algorithm — the classical
$O(n)$ tridiagonal solver.

---

**Contents**

| Section | Topic |
|:--------|:------|
| 0 | Imports and setup |
| 1 | Thomas' algorithm (classical reference) |
| 2 | Analytical training data generation |
| 3 | Transformer architecture |
| 4 | Training loop |
| 5 | Grid-convergence study |
| 6 | Ablation: model depth and width |
| 7 | Generalisation to unseen source functions |
| 8 | Visualisation |
| 9 | Summary table and discussion |

**Problem statement:**

$$-\frac{d^2 u}{dx^2} = f(x), \quad x \in [0,1], \quad u(0)=u(1)=0$$

We use the manufactured source $f(x) = \sin(\pi x)$, for which the exact solution
is $u(x) = \sin(\pi x)/\pi^2$, as the canonical benchmark.
The transformer is trained on *random* combinations of sine modes, so that it
learns to approximate the operator inverse $\mathcal{L}^{-1}$, not just a single
instance of the problem.

## Section 0 — Imports and Setup

In [ ]:
%matplotlib inline
import math
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

SEP = '=' * 68
K_MAX = 8   # maximum Fourier mode used in training data

## Section 1 — Thomas' Algorithm (Classical Reference)

### Finite-Difference Discretisation

With $n$ interior grid points $x_i = ih$, $h = 1/(n+1)$, $i = 1,\ldots,n$,
the second-order central-difference approximation of $-u''$ gives the
symmetric tridiagonal linear system

$$A\mathbf{u} = \mathbf{f}, \qquad
A = \frac{1}{h^2}\begin{pmatrix}2&-1\\-1&2&-1\\&\ddots&\ddots&\ddots\\&&-1&2\end{pmatrix}$$

This discretisation has **second-order truncation error**: $\|u_h - u\|_\infty = O(h^2)$.

### Thomas' Algorithm

Thomas' algorithm is **Gaussian elimination specialised to the tridiagonal
structure**.  It is direct ($O(n)$ operations, no iteration) and numerically
stable for diagonally dominant matrices such as the Poisson matrix.

**Forward sweep** (eliminate the subdiagonal):
$$w_i = \frac{a_i}{b'_{i-1}}, \quad
b'_i = b_i - w_i c_{i-1}, \quad
d'_i = d_i - w_i d'_{i-1}, \qquad i = 1,\ldots,n-1$$

**Back substitution**:
$$x_{n-1} = d'_{n-1}/b'_{n-1}, \quad
x_i = (d'_i - c_i\,x_{i+1})/b'_i, \qquad i=n-2,\ldots,0$$

The Poisson matrix satisfies the strict diagonal-dominance condition
$|b_i| > |a_i| + |c_i|$ everywhere, so no pivoting is needed and the
algorithm is guaranteed to produce the exact solution (up to floating-point
rounding).

In [ ]:
def thomas_solve(n, f_vals):
    """
    Solve  -u'' = f  on n interior points with Dirichlet BCs
    using Thomas' algorithm  (O(n) tridiagonal Gaussian elimination).
    """
    h     = 1.0 / (n + 1)
    main  = np.full(n,    2.0 / h**2)
    upper = np.full(n-1, -1.0 / h**2)
    lower = np.full(n-1, -1.0 / h**2)
    rhs   = f_vals.copy().astype(float)

    b_ = main.copy(); d_ = rhs.copy(); c_ = upper.copy()
    # Forward sweep
    for i in range(1, n):
        w     = lower[i-1] / b_[i-1]
        b_[i] -= w * c_[i-1]
        d_[i] -= w * d_[i-1]
    # Back substitution
    u      = np.zeros(n)
    u[-1]  = d_[-1] / b_[-1]
    for i in range(n-2, -1, -1):
        u[i] = (d_[i] - c_[i] * u[i+1]) / b_[i]
    return u


def interior_grid(n):
    """Interior grid points x_i = i*h, h = 1/(n+1), i = 1,...,n."""
    h = 1.0 / (n + 1)
    return np.linspace(h, 1.0 - h, n)


# Quick demo on n=8
n_demo  = 8
x_demo  = interior_grid(n_demo)
f_demo  = np.sin(np.pi * x_demo)
u_th    = thomas_solve(n_demo, f_demo)
u_ex    = np.sin(np.pi * x_demo) / np.pi**2

print(f'n={n_demo}, h={1/(n_demo+1):.4f}')
print(f'Thomas solution: {np.round(u_th,6)}')
print(f'Exact solution:  {np.round(u_ex,6)}')
print(f'Max error:       {np.max(np.abs(u_th-u_ex)):.4e}')

## Section 2 — Analytical Training Data Generation

### Key Idea: Learn the Operator, Not a Single Solution

We want the transformer to approximate the **operator inverse**
$\mathcal{L}^{-1}: f \mapsto u$, not just to solve one particular instance.
To achieve this we generate thousands of analytically exact $(f, u)$ pairs
by expanding the source function in the eigenbasis of $\mathcal{L}$.

### Eigenfunction Expansion

The eigenfunctions of $\mathcal{L} = -d^2/dx^2$ on $[0,1]$ with Dirichlet
boundary conditions are $\phi_k(x) = \sqrt{2}\sin(k\pi x)$ with eigenvalues
$\lambda_k = (k\pi)^2$.  Therefore, if

$$f(x) = \sum_{k=1}^{K} a_k \sin(k\pi x)$$

then the exact solution is

$$u(x) = \sum_{k=1}^{K} \frac{a_k}{(k\pi)^2} \sin(k\pi x).$$

Each training sample draws $K=8$ random Gaussian amplitudes $\{a_k\}$,
forms both $f$ and $u$ analytically, and normalises by $\|f\|_2$ for
consistent scale.  This gives **zero label noise** — the exact solution
is always available for supervision.

### Out-of-Distribution Generalisation

Section 7 will test the trained model on source functions *outside* the
training distribution (Gaussian bump, polynomial) to measure how well the
model has internalised the structure of $\mathcal{L}^{-1}$.

In [ ]:
def make_sample(n, k_max=K_MAX, rng=None):
    """One training sample: random combination of K_MAX sine modes."""
    if rng is None:
        rng = np.random
    x    = interior_grid(n)
    ks   = np.arange(1, k_max + 1)
    amps = rng.standard_normal(k_max)
    f    = np.sum(amps[:,None] * np.sin(np.pi * ks[:,None] * x[None,:]), axis=0)
    u    = np.sum((amps/(np.pi*ks)**2)[:,None]
                 * np.sin(np.pi * ks[:,None] * x[None,:]), axis=0)
    scale = np.linalg.norm(f) + 1e-8
    return x, f / scale, u / scale


def make_dataset(n, n_samples, k_max=K_MAX, seed=0):
    """Dataset of (f, u) pairs; returns tensors of shape (n_samples, n)."""
    rng    = np.random.default_rng(seed)
    F_list, U_list = [], []
    for _ in range(n_samples):
        x, f, u = make_sample(n, k_max, rng)
        F_list.append(f); U_list.append(u)
    F = torch.tensor(np.array(F_list), dtype=torch.float32)
    U = torch.tensor(np.array(U_list), dtype=torch.float32)
    return F, U


def exact_fourier(x, f_vals, k_max=32):
    """High-accuracy reference: project f onto sine modes, invert exactly."""
    h  = x[1] - x[0]
    ks = np.arange(1, k_max + 1)
    amps = 2.0 * h * np.array([np.sum(f_vals * np.sin(k*np.pi*x)) for k in ks])
    return np.sum((amps/(np.pi*ks)**2)[:,None]
                 * np.sin(np.pi*ks[:,None]*x[None,:]), axis=0)


# Preview a few training samples
x_prev = interior_grid(16)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
rng_prev = np.random.default_rng(7)
for i, ax in enumerate(axes):
    _, f_p, u_p = make_sample(16, rng=rng_prev)
    ax.plot(x_prev, f_p, 'b-', lw=1.5, label='f(x) source')
    ax.plot(x_prev, u_p, 'r--', lw=1.5, label='u(x) solution')
    ax.set(xlabel='x', title=f'Sample {i+1}')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
fig.suptitle('Example training pairs  $(f, u)$ with random Fourier amplitudes',
             fontsize=11)
plt.tight_layout(); plt.show()

## Section 3 — Transformer Architecture

### Design Philosophy

The key observation is that the discrete solution operator is
$u_i = \sum_j G_{ij} f_j$, where $G = A^{-1}$ is the **discrete Green's
function matrix** of the Poisson problem:

$$G_{ij} = \frac{h^2}{n+1}\,\min(i,j)\,(n+1-\max(i,j))$$

This is a tent-shaped, symmetric kernel that is largest on the diagonal and
decays linearly away from it.  The self-attention mechanism in the transformer
computes exactly a similar global weighted sum across all positions:

$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

where $Q,K,V \in \mathbb{R}^{n \times d_k}$ are linear projections of the token
embeddings.  A well-trained head should develop attention weights that
approximate the tent-shaped Green's function kernel.

### Pipeline

```
Input: [x_i, f(x_i)]  ∈  R²   (one 2-vector per interior grid point)
   ↓  Linear(2 → d_model)       token embedding
   + sinusoidal positional encoding  PE(x_i)  [physical coordinate, not index]
   ↓  N × TransformerEncoderLayer  (pre-LayerNorm)
        LayerNorm → MultiHeadAttention → residual
        LayerNorm → FFN(d_model → d_ffn → d_model, GELU) → residual
   ↓  output head: LayerNorm → Linear → GELU → Linear(→ 1)
Output: u_pred_i  ∈  R   (solution value at each interior point)
```

### Positional Encoding

Following Vaswani et al. (2017) the positional encoding is sinusoidal:

$$\text{PE}(p, 2k) = \sin\!\left(\frac{p}{10000^{2k/d}}\right), \qquad
\text{PE}(p, 2k+1) = \cos\!\left(\frac{p}{10000^{2k/d}}\right)$$

Crucially, $p$ is the **physical coordinate** $x_i \in (0,1)$, not the integer
index.  This means the model encodes actual spatial location, which is
important for generalisation: a model trained on $n=32$ points could in
principle evaluate at any grid simply by changing the input coordinates.

### Boundary Conditions

$u(0)=u(1)=0$ are enforced **analytically**: the boundary nodes are simply
not part of the token sequence, so the model only predicts the $n$ interior
values.  No penalty term, no soft constraint — the boundary conditions are
satisfied exactly for every prediction.

### Pre-LayerNorm vs Post-LayerNorm

The original Transformer (Vaswani et al.) used post-LayerNorm: the residual
connection adds the raw sublayer output and *then* normalises.  This can
cause gradient instability in deep networks.  We use **pre-LayerNorm**
(`norm_first=True` in PyTorch): normalise the input *before* passing it to
the sublayer.  This is now standard in large language models and greatly
improves training stability.

In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    """
    Sinusoidal positional encoding (Vaswani et al. 2017).
    'pos' is the physical coordinate x_i in (0,1), not the integer index.
    """
    def __init__(self, d_model: int, max_len: int = 4096):
        super().__init__()
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float()
                        * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, :x.size(1)]


class PoissonTransformer(nn.Module):
    """
    Encoder-only Transformer for the 1-D Poisson operator inverse.

    Each interior grid point is one token with input [x_i, f(x_i)].
    The model predicts u(x_i) for all interior points simultaneously.
    Boundary conditions u(0)=u(1)=0 are enforced analytically.
    """
    def __init__(self, d_model=64, n_heads=4, n_layers=4,
                 d_ffn=256, dropout=0.0):
        super().__init__()
        self.token_embed = nn.Linear(2, d_model)
        self.pos_enc     = SinusoidalPositionalEncoding(d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=d_ffn, dropout=dropout,
            activation='gelu', batch_first=True,
            norm_first=True,   # pre-LN: more stable gradients
        )
        self.encoder     = nn.TransformerEncoder(encoder_layer,
                                                  num_layers=n_layers)
        self.output_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Linear(d_model // 2, 1),
        )
        n_params = sum(p.numel() for p in self.parameters()
                       if p.requires_grad)
        print(f'  PoissonTransformer: d={d_model}, heads={n_heads}, '
              f'layers={n_layers}, d_ffn={d_ffn}, params={n_params:,}')

    def forward(self, x_grid, f_vals):
        """
        x_grid : (batch, n)  spatial coordinates
        f_vals : (batch, n)  source values
        returns  u_pred : (batch, n)  predicted interior solution
        """
        tokens = torch.stack([x_grid, f_vals], dim=-1)  # (B, n, 2)
        h = self.token_embed(tokens)                     # (B, n, d_model)
        h = self.pos_enc(h)
        h = self.encoder(h)
        return self.output_head(h).squeeze(-1)           # (B, n)


# Instantiate the default model to inspect its parameter count
print('Default model (medium config):')
_ = PoissonTransformer(d_model=64, n_heads=4, n_layers=4, d_ffn=256)

## Section 4 — Training Loop

### Optimisation Details

| Hyperparameter | Value | Rationale |
|:---------------|:------|:----------|
| Optimiser | Adam | Adaptive moment estimation, well-suited to non-convex regression |
| Weight decay | $10^{-5}$ | Light $L^2$ regularisation to avoid overfitting |
| Initial LR | $3\times 10^{-4}$ | Standard for Adam on regression tasks |
| LR schedule | Cosine annealing | Smooth decay to 0; avoids abrupt LR drops |
| Batch size | 128 | Balances gradient noise and throughput |
| Gradient clipping | norm $\leq 1.0$ | Prevents occasional large gradient spikes |
| Loss | MSE | Natural for regression; equal weight on all grid points |

### Validation Metric

During training we monitor the **relative $L^2$ error** per sample:

$$\text{rel-}L^2 = \frac{\|u_{\text{pred}} - u_{\text{exact}}\|_2}{\|u_{\text{exact}}\|_2 + \varepsilon}$$

averaged over the validation set.  This is more interpretable than the raw MSE
because it is scale-invariant: a value of 0.01 means the prediction matches
the exact solution to within 1% in the $L^2$ sense.

In [ ]:
def train_model(model, n_grid, n_train=8000, n_val=1000,
                epochs=200, batch_size=128, lr=3e-4, k_max=K_MAX):
    """
    Train PoissonTransformer on analytically generated (f, u) pairs.
    Returns history dict with 'train_loss', 'val_loss', 'val_rel_l2'.
    """
    model.to(device)
    F_tr, U_tr = make_dataset(n_grid, n_train, k_max=k_max, seed=0)
    F_va, U_va = make_dataset(n_grid, n_val,   k_max=k_max, seed=1)
    x_np     = interior_grid(n_grid)
    x_tensor = torch.tensor(x_np, dtype=torch.float32)
    x_tr = x_tensor.unsqueeze(0).expand(n_train, -1).to(device)
    x_va = x_tensor.unsqueeze(0).expand(n_val,   -1).to(device)
    F_tr, U_tr = F_tr.to(device), U_tr.to(device)
    F_va, U_va = F_va.to(device), U_va.to(device)

    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.MSELoss()
    history   = {'train_loss': [], 'val_loss': [], 'val_rel_l2': []}
    t0        = time.time()

    for epoch in range(1, epochs + 1):
        model.train()
        perm      = torch.randperm(n_train, device=device)
        ep_loss   = 0.0; n_batches = 0
        for start in range(0, n_train, batch_size):
            idx  = perm[start:start + batch_size]
            pred = model(x_tr[idx], F_tr[idx])
            loss = criterion(pred, U_tr[idx])
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            ep_loss += loss.item(); n_batches += 1
        scheduler.step()
        history['train_loss'].append(ep_loss / n_batches)

        model.eval()
        with torch.no_grad():
            val_pred = model(x_va, F_va)
            val_loss = criterion(val_pred, U_va).item()
            rel_l2   = ((val_pred - U_va).norm(dim=1)
                        / (U_va.norm(dim=1) + 1e-8)).mean().item()
        history['val_loss'].append(val_loss)
        history['val_rel_l2'].append(rel_l2)

        if epoch % 50 == 0 or epoch == 1:
            print(f'  ep {epoch:4d}/{epochs}  '
                  f'train={ep_loss/n_batches:.3e}  '
                  f'val={val_loss:.3e}  '
                  f'rel-L2={rel_l2:.3e}  '
                  f'lr={scheduler.get_last_lr()[0]:.1e}')

    print(f'  Done in {time.time()-t0:.1f}s')
    return history

## Section 5 — Grid-Convergence Study

We train a **separate model** for each grid size $n \in \{8, 16, 32, 64\}$
and measure the max-norm error on the canonical test case
$f(x)=\sin(\pi x)$, $u(x)=\sin(\pi x)/\pi^2$.

**Thomas' algorithm** achieves second-order convergence:
$\|u_h - u\|_\infty = O(h^2) = O(1/(n+1)^2)$, confirmed by the slope of
$-2$ on the log-log convergence plot.

**The transformer** has two error sources:
1. The **learned approximation error** — how well the model has internalised
   the operator structure from the training data.
2. The implicit **finite-difference error** $O(h^2)$ inherited from the fact
   that the training labels are computed on a discrete grid.

At larger $n$ the model must learn a finer mapping, which may require more
training data or a larger architecture.  The convergence study reveals how
quickly the transformer's error decreases with grid refinement.

> **Note:** this cell trains four models sequentially.  On a CPU this takes
> several minutes per model.  Reduce `epochs` in `TRAIN_CFG` for a quick test.

In [ ]:
def test_source(x): return np.sin(np.pi * x)
def test_exact(x):  return np.sin(np.pi * x) / np.pi**2

grid_sizes_cv = [8, 16, 32, 64]
MODEL_CFG = dict(d_model=64, n_heads=4, n_layers=4, d_ffn=256, dropout=0.0)
TRAIN_CFG = dict(n_train=8000, n_val=1000, epochs=300,
                 batch_size=128, lr=3e-4, k_max=K_MAX)

convergence_results = {}

for n_cv in grid_sizes_cv:
    print(f'\n{"─"*55}')
    print(f'  n = {n_cv}  (h = {1/(n_cv+1):.5f})')
    print(f'{"─"*55}')
    x_cv  = interior_grid(n_cv)
    f_cv  = test_source(x_cv)
    u_ex  = test_exact(x_cv)
    u_th  = thomas_solve(n_cv, f_cv)
    err_th = np.max(np.abs(u_th - u_ex))

    print(f'  Training transformer (n={n_cv})...')
    model_cv = PoissonTransformer(**MODEL_CFG)
    hist_cv  = train_model(model_cv, n_cv, **TRAIN_CFG)

    model_cv.eval()
    with torch.no_grad():
        x_t  = torch.tensor(x_cv, dtype=torch.float32).unsqueeze(0).to(device)
        f_t  = torch.tensor(f_cv, dtype=torch.float32).unsqueeze(0).to(device)
        fsc  = f_t.norm() + 1e-8
        u_tr = model_cv(x_t, f_t/fsc).squeeze(0).cpu().numpy() * fsc.item()

    err_tr = np.max(np.abs(u_tr - u_ex))
    convergence_results[n_cv] = {
        'x': x_cv, 'u_exact': u_ex, 'u_thomas': u_th, 'u_transf': u_tr,
        'err_th': err_th, 'err_tr': err_tr,
        'err_l2_tr': np.sqrt(np.mean((u_tr-u_ex)**2)),
        'history': hist_cv,
    }
    print(f'  Thomas  max err : {err_th:.4e}')
    print(f'  Transf. max err : {err_tr:.4e}')

## Section 6 — Ablation: Model Depth and Width

We fix $n = 32$ and train four model configurations ranging from tiny
(~1k parameters) to large (~200k parameters) to understand how model
capacity affects accuracy.

| Config | $d_{\text{model}}$ | Heads | Layers | $d_{\text{ffn}}$ |
|:-------|:-------:|:-----:|:------:|:-------:|
| tiny   | 16 | 2 | 2 | 64 |
| small  | 32 | 2 | 2 | 128 |
| medium | 64 | 4 | 4 | 256 |
| large  | 128 | 4 | 6 | 512 |

The ablation answers the question: **how much transformer capacity is needed
to match Thomas' $O(h^2)$ accuracy?**  For the Poisson problem the operator
inverse is smooth and the Green's function is a simple tent function, so
relatively modest models should suffice.

In [ ]:
ABLATION_N      = 32
ABLATION_EPOCHS = 200

x_ab    = interior_grid(ABLATION_N)
f_ab    = test_source(x_ab)
u_ex_ab = test_exact(x_ab)
u_th_ab = thomas_solve(ABLATION_N, f_ab)

ablation_configs = [
    dict(label='tiny',   d_model=16,  n_heads=2, n_layers=2, d_ffn=64),
    dict(label='small',  d_model=32,  n_heads=2, n_layers=2, d_ffn=128),
    dict(label='medium', d_model=64,  n_heads=4, n_layers=4, d_ffn=256),
    dict(label='large',  d_model=128, n_heads=4, n_layers=6, d_ffn=512),
]

ablation_results = []
for cfg in ablation_configs:
    label = cfg.pop('label')
    print(f'\n  Config: {label}')
    model_ab = PoissonTransformer(**cfg, dropout=0.0)
    hist_ab  = train_model(model_ab, ABLATION_N,
                           n_train=6000, n_val=800,
                           epochs=ABLATION_EPOCHS, batch_size=128, lr=3e-4)
    model_ab.eval()
    with torch.no_grad():
        x_t = torch.tensor(x_ab, dtype=torch.float32).unsqueeze(0).to(device)
        f_t = torch.tensor(f_ab, dtype=torch.float32).unsqueeze(0).to(device)
        fsc = f_t.norm() + 1e-8
        u_ab_p = model_ab(x_t, f_t/fsc).squeeze(0).cpu().numpy() * fsc.item()
    err  = np.max(np.abs(u_ab_p - u_ex_ab))
    npar = sum(p.numel() for p in model_ab.parameters())
    ablation_results.append({
        'label': label, 'n_params': npar, 'err_max': err,
        'val_l2': hist_ab['val_rel_l2'][-1], 'history': hist_ab,
        'd_model': cfg['d_model'], 'n_layers': cfg['n_layers'],
    })
    cfg['label'] = label
    print(f'  Max err: {err:.4e}  (Thomas ref: {np.max(np.abs(u_th_ab-u_ex_ab)):.4e})')

## Section 7 — Generalisation to Unseen Source Functions

A well-trained model should have internalised the structure of $\mathcal{L}^{-1}$,
not merely memorised training examples.  We test this by evaluating on four
qualitatively different source functions:

| Source | Type | Exact solution |
|:-------|:-----|:---------------|
| $\sin(\pi x)$ | In-distribution | $\sin(\pi x)/\pi^2$ |
| $\sin(6\pi x)$ | Higher Fourier mode | $\sin(6\pi x)/(6\pi)^2$ |
| $e^{-50(x-0.5)^2}$ | Gaussian bump (OOD) | Thomas (reference) |
| $x(1-x)$ | Polynomial (OOD) | $x(1-x^2)/6$ |

The **ratio** (transformer error) / (Thomas error) tells us how far the neural
solver is from the classical reference.  A ratio of 1 means the two are equally
accurate; a ratio $\gg 1$ signals that the model is struggling with this source.

In [ ]:
# Re-train a fresh n=32 model for the generalisation section
print('Training generalisation model (n=32)...')
model_gen = PoissonTransformer(**MODEL_CFG)
train_model(model_gen, 32, **TRAIN_CFG)

x_gen   = interior_grid(32)
ood_cases = {
    'sin(pi*x)  [in-dist]' : (
        lambda x: np.sin(np.pi * x),
        lambda x: np.sin(np.pi * x) / np.pi**2
    ),
    'sin(6pi*x) [higher k]': (
        lambda x: np.sin(6*np.pi*x),
        lambda x: np.sin(6*np.pi*x) / (6*np.pi)**2
    ),
    'Gaussian bump   [OOD]': (
        lambda x: np.exp(-50*(x-0.5)**2),
        None   # use Thomas as reference
    ),
    'Poly x(1-x)     [OOD]': (
        lambda x: x*(1-x),
        lambda x: x*(1-x**2)/6
    ),
}

gen_results = {}
print(f'\n  {"Source":26s}  {"Thomas err":>12}  {"Transf err":>12}  {"Ratio":>8}')
print('  ' + '-'*64)
for name, (f_fn, u_fn) in ood_cases.items():
    f_g   = f_fn(x_gen)
    u_ref = u_fn(x_gen) if u_fn is not None else thomas_solve(32, f_g)
    u_th_g = thomas_solve(32, f_g)
    model_gen.eval()
    with torch.no_grad():
        x_t = torch.tensor(x_gen, dtype=torch.float32).unsqueeze(0).to(device)
        f_t = torch.tensor(f_g,   dtype=torch.float32).unsqueeze(0).to(device)
        fsc = f_t.norm() + 1e-8
        u_tr_g = model_gen(x_t, f_t/fsc).squeeze(0).cpu().numpy() * fsc.item()
    err_th = np.max(np.abs(u_th_g - u_ref))
    err_tr = np.max(np.abs(u_tr_g - u_ref))
    gen_results[name] = {
        'f': f_g, 'u_ref': u_ref, 'u_thomas': u_th_g, 'u_transf': u_tr_g,
        'err_th': err_th, 'err_tr': err_tr,
    }
    print(f'  {name:26s}  {err_th:>12.4e}  {err_tr:>12.4e}  {err_tr/(err_th+1e-12):>8.2f}x')

## Section 8 — Visualisation

The 8-panel figure summarises all results:

**Row 1:**
1. Solution comparison for $n=32$: exact, Thomas, transformer
2. Grid-convergence log-log plot with $O(h^2)$ reference slope
3. Training curves (train MSE, val MSE, val relative-$L^2$)
4. Ablation: max error vs parameter count

**Row 2:**
5. Pointwise error $|u_{\text{transf}} - u_{\text{exact}}|$ vs position
6. Residual histogram comparing transformer and Thomas at $n=32$
7. Generalisation: solution curves for three OOD test functions
8. Architecture diagram

In [ ]:
fig = plt.figure(figsize=(20, 14))
fig.suptitle(
    "Transformer Solver vs Thomas' Algorithm — 1-D Poisson Equation\n"
    r"$-u''=f(x)$, $u(0)=u(1)=0$",
    fontsize=14, fontweight='bold'
)
gs  = gridspec.GridSpec(2, 4, hspace=0.50, wspace=0.42)
n_ref = 32

# ── Panel 1: Solution comparison ─────────────────────────────────────────────
ax = fig.add_subplot(gs[0, 0])
if n_ref in convergence_results:
    r   = convergence_results[n_ref]
    x_d = np.linspace(0, 1, 400)
    ax.plot(x_d, test_exact(x_d), 'k-', lw=2.5,
            label=r'Exact $\sin(\pi x)/\pi^2$')
    ax.plot(np.concatenate([[0], r['x'], [1]]),
            np.concatenate([[0], r['u_thomas'], [0]]),
            'bs--', ms=5, lw=1.5, label=f"Thomas ({r['err_th']:.1e})")
    ax.plot(np.concatenate([[0], r['x'], [1]]),
            np.concatenate([[0], r['u_transf'], [0]]),
            'r^:', ms=5, lw=1.5, label=f"Transformer ({r['err_tr']:.1e})")
    ax.set(xlabel='$x$', ylabel='$u(x)$',
           title=f'Solution  $n={n_ref}$, $f=\\sin(\\pi x)$')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# ── Panel 2: Grid convergence ─────────────────────────────────────────────────
ax = fig.add_subplot(gs[0, 1])
ns_cv  = sorted(convergence_results)
hs_cv  = [1/(n+1) for n in ns_cv]
th_err = [convergence_results[n]['err_th'] for n in ns_cv]
tr_err = [convergence_results[n]['err_tr'] for n in ns_cv]
ax.loglog(hs_cv, th_err, 'bs-',  lw=2, ms=8, label="Thomas'")
ax.loglog(hs_cv, tr_err, 'r^--', lw=2, ms=8, label='Transformer')
h0, e0 = hs_cv[0], th_err[0]
ax.loglog(hs_cv, [e0*(h/h0)**2 for h in hs_cv], 'k:', lw=1.5,
          alpha=0.6, label=r'$O(h^2)$')
ax.set(xlabel='$h$', ylabel=r'$\|u_{\rm pred}-u_{\rm ex}\|_\infty$',
       title='Grid convergence')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3, which='both')

# ── Panel 3: Training curves ──────────────────────────────────────────────────
ax = fig.add_subplot(gs[0, 2])
if n_ref in convergence_results:
    hist = convergence_results[n_ref]['history']
    ep   = range(1, len(hist['train_loss']) + 1)
    ax.semilogy(ep, hist['train_loss'], 'b-',  lw=2, label='Train MSE')
    ax.semilogy(ep, hist['val_loss'],   'r--', lw=2, label='Val MSE')
    ax.semilogy(ep, hist['val_rel_l2'], 'g:',  lw=2, label='Val rel-L2')
    ax.set(xlabel='Epoch', ylabel='Loss',
           title=f'Training curves  $n={n_ref}$')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# ── Panel 4: Ablation ─────────────────────────────────────────────────────────
ax = fig.add_subplot(gs[0, 3])
if ablation_results:
    npar = [r['n_params'] for r in ablation_results]
    errs = [r['err_max']  for r in ablation_results]
    ax.loglog(npar, errs, 'mo-', lw=2, ms=8)
    for r in ablation_results:
        ax.annotate(r['label'], (r['n_params'], r['err_max']),
                    textcoords='offset points', xytext=(5,3), fontsize=8)
    ax.axhline(np.max(np.abs(u_th_ab-u_ex_ab)), color='steelblue',
               ls='--', lw=1.5, label="Thomas' err")
    ax.set(xlabel='Parameters', ylabel='Max error',
           title=f'Ablation  $n={ABLATION_N}$')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3, which='both')

# ── Panel 5: Pointwise error ──────────────────────────────────────────────────
ax = fig.add_subplot(gs[1, 0])
cols = ['steelblue', 'firebrick', 'seagreen', 'darkorange']
for col, n_pt in zip(cols, ns_cv):
    r = convergence_results[n_pt]
    ax.semilogy(r['x'], np.abs(r['u_transf']-r['u_exact']),
                '-', color=col, lw=1.5, label=f'$n={n_pt}$')
ax.set(xlabel='$x$', ylabel=r'$|u_{\rm transf}-u_{\rm ex}|$',
       title='Transformer pointwise error')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# ── Panel 6: Residual histogram ───────────────────────────────────────────────
ax = fig.add_subplot(gs[1, 1])
if n_ref in convergence_results:
    r = convergence_results[n_ref]
    ax.hist(r['u_transf']-r['u_exact'], bins=25, alpha=0.6, color='firebrick',
            label=f"Transf. (σ={np.std(r['u_transf']-r['u_exact']):.2e})")
    ax.hist(r['u_thomas']-r['u_exact'], bins=25, alpha=0.6, color='steelblue',
            label=f"Thomas' (σ={np.std(r['u_thomas']-r['u_exact']):.2e})")
    ax.axvline(0, color='k', lw=1.5, ls='--')
    ax.set(xlabel='Residual', ylabel='Count',
           title=f'Residual distribution  $n={n_ref}$')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3, axis='y')

# ── Panel 7: Generalisation ───────────────────────────────────────────────────
ax = fig.add_subplot(gs[1, 2])
styles = ['-', '--', ':']
for ls, name in zip(styles, list(ood_cases)[:3]):
    gr = gen_results.get(name, {})
    if not gr: continue
    ax.plot(x_gen, gr['u_ref'],    'k',          lw=2,   ls=ls, alpha=0.7)
    ax.plot(x_gen, gr['u_transf'], 'firebrick',  lw=1.5, ls=ls)
    ax.plot(x_gen, gr['u_thomas'], 'steelblue',  lw=1.5, ls=ls)
proxy = [Line2D([0],[0], color='k',         lw=2,   label='Exact/ref'),
         Line2D([0],[0], color='firebrick', lw=1.5, label='Transformer'),
         Line2D([0],[0], color='steelblue', lw=1.5, label="Thomas'")]
ax.legend(handles=proxy, fontsize=8)
ax.set(xlabel='$x$', ylabel='$u(x)$',
       title='Generalisation (n=32, 3 test functions)')
ax.grid(True, alpha=0.3)

# ── Panel 8: Architecture diagram ─────────────────────────────────────────────
ax = fig.add_subplot(gs[1, 3])
ax.axis('off')
diagram = (
    'POISSONTRANSFORMER\n'
    '─────────────────────────────\n\n'
    'Token: [x_i, f(x_i)]  in R^2\n\n'
    'Linear(2 → d_model)\n'
    '+ Sinusoidal PE(x_i)\n\n'
    'N x EncoderLayer (pre-LN):\n'
    '  LN → MultiHeadAttn → +\n'
    '  LN → FFN(GELU)    → +\n\n'
    'Output head:\n'
    '  LN → Linear → GELU → Linear(1)\n\n'
    'BCs: u(0)=u(1)=0 exact\n'
    '(boundary nodes are not tokens)\n\n'
    'Attn weight A_ij approx G_ij\n'
    'G_ij = h^2 min(i,j)(n+1-max(i,j))/(n+1)'
)
ax.text(0.03, 0.97, diagram, transform=ax.transAxes,
        fontsize=8.5, va='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))
ax.set_title('Architecture', fontweight='bold', fontsize=10)

plt.savefig('transformer_poisson.png', dpi=140, bbox_inches='tight')
plt.show()
print('Figure saved to transformer_poisson.png')

## Section 9 — Summary Table and Discussion

### Results

In [ ]:
print(f'GRID CONVERGENCE  (||u_pred - u_exact||_inf)')
print(f'  {"n":>6}  {"h":>9}  {"Thomas":>14}  {"Transformer":>14}  {"Ratio":>8}')
print('  ' + '-'*58)
for n_cv in sorted(convergence_results):
    r = convergence_results[n_cv]; h = 1/(n_cv+1)
    ratio = r['err_tr'] / (r['err_th'] + 1e-15)
    print(f'  {n_cv:>6}  {h:>9.5f}  {r["err_th"]:>14.4e}  '
          f'{r["err_tr"]:>14.4e}  {ratio:>8.2f}x')

print()
print(f'ABLATION (n={ABLATION_N})')
print(f'  {"Config":>8}  {"d_model":>8}  {"Layers":>7}  '
      f'{"Params":>10}  {"Max err":>12}  {"Val rel-L2":>12}')
print('  ' + '-'*62)
for r in ablation_results:
    print(f'  {r["label"]:>8}  {r["d_model"]:>8}  {r["n_layers"]:>7}  '
          f'{r["n_params"]:>10,}  {r["err_max"]:>12.4e}  {r["val_l2"]:>12.4e}')

print()
print(f'GENERALISATION (n=32)')
print(f'  {"Source":26s}  {"Thomas err":>12}  {"Transf err":>12}  {"Ratio":>8}')
print('  ' + '-'*64)
for name, gr in gen_results.items():
    ratio = gr['err_tr'] / (gr['err_th'] + 1e-12)
    print(f'  {name:26s}  {gr["err_th"]:>12.4e}  {gr["err_tr"]:>12.4e}  {ratio:>8.2f}x')

### Discussion

**Thomas' algorithm** is unbeatable for this specific problem:
- $O(n)$ time, zero training cost, exact up to floating-point rounding
- Guaranteed $O(h^2)$ convergence as $n\to\infty$
- Requires only knowledge of the tridiagonal structure — no data

**The Transformer solver** has different strengths and limitations:

| Aspect | Thomas | Transformer |
|:-------|:-------|:------------|
| Training cost | None | $O(N_{\rm train}\times\text{epochs}\times n^2)$ |
| Inference cost | $O(n)$ | $O(n^2 d)$ per forward pass |
| Error type | $O(h^2)$ FD only | Learned error + implicit $O(h^2)$ FD |
| Convergence | Guaranteed | Problem-dependent |
| Generality | Tridiagonal only | Any $f$ on trained grid |
| Physics knowledge | Grid spacing $h$ | Only grid positions |

### Attention Patterns and the Green's Function

The discrete 1-D Poisson Green's function is

$$G_{ij} = \frac{h^2}{n+1}\,\min(i,j)\,(n+1-\max(i,j))$$

This is a tent-shaped, symmetric kernel: it is maximal on the diagonal and
decays linearly away from it.  A well-trained transformer head should develop
attention weights that reflect this structure: each token $i$ should attend
most strongly to nearby tokens $j \approx i$, with weight decaying as
$|i-j|$ increases — but maintaining *global* support because the Green's
function is non-zero for all $(i,j)$.

### When Does a Transformer Solver Make Sense?

The transformer approach becomes genuinely advantageous in settings where:
1. The operator $\mathcal{L}$ is **non-constant** (variable coefficients) or
   **non-standard** (fractional, non-local), making factorisation expensive.
2. The same operator must be solved for **many different right-hand sides** —
   amortising the training cost over many queries.
3. One needs **surrogates** for expensive PDE solves inside optimisation loops
   (e.g., design optimisation, uncertainty quantification).
4. The operator itself changes with parameters (parametric PDEs), and the
   model can be conditioned on those parameters.

For the constant-coefficient 1-D Poisson problem, Thomas' algorithm is the
clear winner.  The transformer is presented here as a didactic demonstration
of neural operator learning in a controlled setting where the ground truth is
always available.